# Building a Custom POMDP Environment

This notebook walks through building a POMDP environment from scratch and running planners on it. By the end, you will understand every abstract method you need to implement and how the framework uses them.

**What's Covered:**
- Anatomy of an existing environment (TigerPOMDP)
- Building a complete custom environment: the **Weather POMDP**
- Running POMCP and PFT_DPW on the custom environment
- Adding custom environment metrics
- Brief notes on continuous-space environments

## 1. Anatomy of an Environment

Every POMDP environment in POMDPPlanners implements the `Environment` base class (or `DiscreteActionsEnvironment` for discrete action spaces). Let's inspect the TigerPOMDP to see what methods are involved and what they return.

In [1]:
import numpy as np
np.random.seed(42)

from POMDPPlanners.environments.tiger_pomdp import TigerPOMDP

tiger = TigerPOMDP(discount_factor=0.95)

# Space info tells planners what kind of spaces to expect
print(f"Space info: {tiger.space_info}")
print(f"Actions: {tiger.get_actions()}")
print()

# State transition model: given (state, action), sample next state
transition = tiger.state_transition_model(state="tiger_left", action="listen")
print(f"Transition sample (listen): {transition.sample()}")
print(f"Transition probability (tiger_left | listen): {transition.probability(['tiger_left'])}")
print()

# Observation model: given (next_state, action), sample observation
obs_model = tiger.observation_model(next_state="tiger_left", action="listen")
print(f"Observation sample: {obs_model.sample()}")
print(f"Observation P(hear_left | tiger_left, listen): {obs_model.probability(['hear_left'])}")
print()

# Reward function
print(f"Reward(tiger_left, listen) = {tiger.reward('tiger_left', 'listen')}")
print(f"Reward(tiger_left, open_left) = {tiger.reward('tiger_left', 'open_left')}")
print(f"Reward(tiger_left, open_right) = {tiger.reward('tiger_left', 'open_right')}")
print()

# Convenience method: sample_next_step combines everything
next_state, observation, reward = tiger.sample_next_step("tiger_left", "listen")
print(f"sample_next_step: next_state={next_state}, obs={observation}, reward={reward}")

2026-02-25 13:02:42,606 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:296 - Initializing TigerPOMDP environment with discount factor 0.95


Space info: SpaceInfo(action_space=<SpaceType.DISCRETE: 'discrete'>, observation_space=<SpaceType.DISCRETE: 'discrete'>)
Actions: ['listen', 'open_left', 'open_right']

Transition sample (listen): ['tiger_left']
Transition probability (tiger_left | listen): [1.]

Observation sample: ['hear_left']
Observation P(hear_left | tiger_left, listen): [0.85]

Reward(tiger_left, listen) = -1.0
Reward(tiger_left, open_left) = -100.0
Reward(tiger_left, open_right) = 10.0

sample_next_step: next_state=tiger_left, obs=hear_right, reward=-1.0


## 2. Building a Custom Environment: Weather POMDP

Let's build a simple **Weather POMDP** from scratch:

- **States**: `sunny`, `rainy` — the true weather
- **Actions**: `setup_outdoor`, `setup_indoor`, `check_forecast`
- **Observations**: `forecast_sunny`, `forecast_rainy`

**Dynamics**:
- Weather persists with 70% probability (sunny stays sunny, rainy stays rainy)
- Forecast is 80% accurate
- Checking the forecast costs -1, setting up outdoor in sunny weather gives +10, setting up outdoor in rain gives -20, setting up indoor always gives +2

We start by defining the **StateTransitionModel** and **ObservationModel**, then assemble the full environment.

### Step A: Transition and Observation Models

In [2]:
import numpy as np
np.random.seed(42)

from typing import Any, List
from POMDPPlanners.core.environment import StateTransitionModel, ObservationModel
from POMDPPlanners.core.distributions import DiscreteDistribution

WEATHER_STATES = ["sunny", "rainy"]
WEATHER_ACTIONS = ["setup_outdoor", "setup_indoor", "check_forecast"]
WEATHER_OBSERVATIONS = ["forecast_sunny", "forecast_rainy"]


class WeatherStateTransition(StateTransitionModel):
    """Weather persists with 70% probability regardless of action."""

    def __init__(self, state: str, action: str):
        super().__init__(state=state, action=action)
        persist = 0.70
        if state == "sunny":
            self._probs = np.array([persist, 1.0 - persist])
        else:
            self._probs = np.array([1.0 - persist, persist])
        self._dist = DiscreteDistribution(values=WEATHER_STATES, probs=self._probs)

    def sample(self, n_samples: int = 1) -> List[str]:
        return self._dist.sample(n_samples)

    def probability(self, values: List[Any]) -> np.ndarray:
        return self._dist.probability(values)


class WeatherObservation(ObservationModel):
    """Forecast is 80% accurate."""

    def __init__(self, next_state: str, action: str):
        super().__init__(next_state=next_state, action=action)
        accuracy = 0.80
        if next_state == "sunny":
            self._probs = np.array([accuracy, 1.0 - accuracy])
        else:
            self._probs = np.array([1.0 - accuracy, accuracy])
        self._dist = DiscreteDistribution(values=WEATHER_OBSERVATIONS, probs=self._probs)

    def sample(self, n_samples: int = 1) -> List[str]:
        return self._dist.sample(n_samples)

    def probability(self, values: List[Any]) -> np.ndarray:
        return self._dist.probability(values)


# Test the models
trans = WeatherStateTransition(state="sunny", action="check_forecast")
print(f"Transition from sunny: {trans.sample(5)}")
print(f"P(sunny | sunny) = {trans.probability(['sunny'])[0]:.2f}")
print(f"P(rainy | sunny) = {trans.probability(['rainy'])[0]:.2f}")
print()

obs = WeatherObservation(next_state="sunny", action="check_forecast")
print(f"Observations given sunny: {obs.sample(5)}")
print(f"P(forecast_sunny | sunny) = {obs.probability(['forecast_sunny'])[0]:.2f}")
print(f"P(forecast_rainy | sunny) = {obs.probability(['forecast_rainy'])[0]:.2f}")

Transition from sunny: ['sunny', 'rainy', 'rainy', 'sunny', 'sunny']
P(sunny | sunny) = 0.70
P(rainy | sunny) = 0.30

Observations given sunny: ['forecast_sunny', 'forecast_sunny', 'forecast_rainy', 'forecast_sunny', 'forecast_sunny']
P(forecast_sunny | sunny) = 0.80
P(forecast_rainy | sunny) = 0.20


### Step B: The Full Environment

Now we assemble the `WeatherPOMDP` class. A `DiscreteActionsEnvironment` requires these abstract methods:

| Method | Purpose |
|---|---|
| `state_transition_model(state, action)` | Return a `StateTransitionModel` |
| `observation_model(next_state, action)` | Return an `ObservationModel` |
| `reward(state, action)` | Return immediate reward (float) |
| `is_terminal(state)` | Return whether the state is terminal |
| `initial_state_dist()` | Return the prior distribution over states |
| `initial_observation_dist()` | Return the initial observation distribution |
| `get_actions()` | Return the list of actions |
| `is_equal_observation(observation1, observation2)` | Compare two observations |

In [3]:
from POMDPPlanners.core.environment import (
    DiscreteActionsEnvironment,
    SpaceInfo,
    SpaceType,
)
from POMDPPlanners.core.distributions import Distribution
from pathlib import Path
from POMDPPlanners.core.simulation import History


class WeatherPOMDP(DiscreteActionsEnvironment):
    """Simple Weather POMDP: decide whether to set up outdoors or indoors."""

    def __init__(self, discount_factor: float = 0.95):
        space_info = SpaceInfo(
            action_space=SpaceType.DISCRETE,
            observation_space=SpaceType.DISCRETE,
        )
        super().__init__(
            discount_factor=discount_factor,
            name="WeatherPOMDP",
            space_info=space_info,
            reward_range=(-20.0, 10.0),
        )

    def state_transition_model(self, state: str, action: str) -> StateTransitionModel:
        return WeatherStateTransition(state=state, action=action)

    def observation_model(self, next_state: str, action: str) -> ObservationModel:
        return WeatherObservation(next_state=next_state, action=action)

    def reward(self, state: str, action: str) -> float:
        if action == "check_forecast":
            return -1.0
        if action == "setup_indoor":
            return 2.0
        # setup_outdoor
        return 10.0 if state == "sunny" else -20.0

    def is_terminal(self, state: str) -> bool:
        return False  # Infinite horizon

    def initial_state_dist(self) -> Distribution:
        return DiscreteDistribution(
            values=WEATHER_STATES, probs=np.array([0.5, 0.5])
        )

    def initial_observation_dist(self) -> Distribution:
        return DiscreteDistribution(
            values=WEATHER_OBSERVATIONS, probs=np.array([0.5, 0.5])
        )

    def get_actions(self) -> List[Any]:
        return WEATHER_ACTIONS

    def is_equal_observation(self, observation1: Any, observation2: Any) -> bool:
        return observation1 == observation2


# Test the environment
weather_env = WeatherPOMDP(discount_factor=0.95)
print(f"Actions: {weather_env.get_actions()}")
print(f"Space info: {weather_env.space_info}")
print()

state = weather_env.initial_state_dist().sample()[0]
print(f"Initial state: {state}")

for action in WEATHER_ACTIONS:
    next_state, obs, reward = weather_env.sample_next_step(state, action)
    print(f"  Action={action}: next_state={next_state}, obs={obs}, reward={reward}")

2026-02-25 13:02:43,162 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:296 - Initializing WeatherPOMDP environment with discount factor 0.95


Actions: ['setup_outdoor', 'setup_indoor', 'check_forecast']
Space info: SpaceInfo(action_space=<SpaceType.DISCRETE: 'discrete'>, observation_space=<SpaceType.DISCRETE: 'discrete'>)

Initial state: sunny
  Action=setup_outdoor: next_state=rainy, obs=forecast_rainy, reward=10.0
  Action=setup_indoor: next_state=sunny, obs=forecast_sunny, reward=2.0
  Action=check_forecast: next_state=sunny, obs=forecast_sunny, reward=-1.0


## 3. Running a Planner on WeatherPOMDP

Now that the environment is defined, any planner in POMDPPlanners can be used with it. Let's run **POMCP** for 15 steps and look at the agent's decisions.

In [4]:
np.random.seed(42)

from POMDPPlanners.planners.mcts_planners.pomcp import POMCP
from POMDPPlanners.core.belief import get_initial_belief
from POMDPPlanners.simulations.episodes import run_episode
from POMDPPlanners.core.simulation import history_to_discounted_return_value

weather_env = WeatherPOMDP(discount_factor=0.95)
belief = get_initial_belief(weather_env, n_particles=50, resampling=True)

pomcp = POMCP(
    environment=weather_env,
    discount_factor=0.95,
    depth=10,
    exploration_constant=10.0,
    name="POMCP_Weather",
    n_simulations=200,
)

history = run_episode(
    environment=weather_env,
    policy=pomcp,
    initial_belief=belief,
    num_steps=15,
    logger=None,
)

print(f"{'Step':<6} {'State':<10} {'Action':<18} {'Obs':<18} {'Reward'}")
print("-" * 60)
for i, step in enumerate(history.history):
    print(f"{i:<6} {step.state:<10} {step.action:<18} {step.observation:<18} {step.reward}")

ret = history_to_discounted_return_value(history)
print(f"\nDiscounted return: {ret:.2f}")

/home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:27: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/kobi/Documents/github/POMDPPlanners/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-25 13:02:45,600 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:296 - Initializing WeatherPOMDP environment with discount factor 0.95
2026-02-25 13:02:45,601 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_Weather (debug=False)


Step   State      Action             Obs                Reward
------------------------------------------------------------
0      rainy      check_forecast     forecast_rainy     -1.0
1      rainy      check_forecast     forecast_rainy     -1.0
2      sunny      check_forecast     forecast_sunny     -1.0
3      sunny      setup_indoor       forecast_rainy     2.0
4      rainy      setup_indoor       forecast_rainy     2.0
5      rainy      setup_indoor       forecast_sunny     2.0
6      sunny      setup_outdoor      forecast_rainy     10.0
7      sunny      setup_outdoor      forecast_rainy     10.0
8      sunny      check_forecast     forecast_rainy     -1.0
9      sunny      setup_indoor       forecast_sunny     2.0
10     sunny      check_forecast     forecast_sunny     -1.0
11     sunny      check_forecast     forecast_sunny     -1.0
12     sunny      setup_outdoor      forecast_rainy     10.0
13     rainy      check_forecast     forecast_rainy     -1.0
14     sunny      setup_in

## 4. Also Run PFT_DPW

PFT_DPW works on any environment when given a `DiscreteActionSampler`. Let's compare its behavior.

In [5]:
np.random.seed(42)

from POMDPPlanners.planners.mcts_planners.pft_dpw import PFT_DPW
from POMDPPlanners.utils.action_samplers import DiscreteActionSampler

weather_env = WeatherPOMDP(discount_factor=0.95)
belief = get_initial_belief(weather_env, n_particles=50, resampling=True)

pft = PFT_DPW(
    environment=weather_env,
    discount_factor=0.95,
    depth=10,
    name="PFT_DPW_Weather",
    action_sampler=DiscreteActionSampler(actions=WEATHER_ACTIONS),
    n_simulations=200,
    exploration_constant=10.0,
)

history_pft = run_episode(
    environment=weather_env,
    policy=pft,
    initial_belief=belief,
    num_steps=15,
    logger=None,
)

ret_pft = history_to_discounted_return_value(history_pft)
print(f"PFT_DPW discounted return: {ret_pft:.2f}")
print()

# Show action choices
actions_chosen = [step.action for step in history_pft.history]
print(f"Actions chosen: {actions_chosen}")

2026-02-25 13:02:48,229 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:296 - Initializing WeatherPOMDP environment with discount factor 0.95
2026-02-25 13:02:48,230 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: PFT_DPW_Weather (debug=False)


PFT_DPW discounted return: -2.14

Actions chosen: ['check_forecast', 'setup_outdoor', 'setup_outdoor', 'check_forecast', 'check_forecast', 'setup_indoor', 'check_forecast', 'setup_indoor', 'check_forecast', 'setup_indoor', 'setup_indoor', 'setup_indoor', 'check_forecast', 'setup_indoor', 'setup_outdoor']


## 5. Adding Custom Metrics

Environments can report custom metrics by overriding `get_metric_names()` and `compute_metrics()`. These metrics are computed across a batch of episode histories and are useful for simulation studies.

Let's track:
- **outdoor_success_rate**: How often `setup_outdoor` was chosen in sunny weather
- **avg_forecasts_per_episode**: Average number of `check_forecast` actions per episode

In [6]:
from POMDPPlanners.core.simulation import MetricValue


class WeatherPOMDPWithMetrics(WeatherPOMDP):
    """WeatherPOMDP with custom performance metrics."""

    def get_metric_names(self) -> List[str]:
        return ["outdoor_success_rate", "avg_forecasts_per_episode"]

    def compute_metrics(self, histories: List[History]) -> List[MetricValue]:
        outdoor_attempts = 0
        outdoor_successes = 0
        total_forecasts = 0

        for hist in histories:
            for step in hist.history:
                if step.action == "setup_outdoor":
                    outdoor_attempts += 1
                    if step.state == "sunny":
                        outdoor_successes += 1
                if step.action == "check_forecast":
                    total_forecasts += 1

        success_rate = outdoor_successes / max(outdoor_attempts, 1)
        avg_forecasts = total_forecasts / max(len(histories), 1)

        return [
            MetricValue(
                name="outdoor_success_rate",
                value=success_rate,
                lower_confidence_bound=success_rate,
                upper_confidence_bound=success_rate,
            ),
            MetricValue(
                name="avg_forecasts_per_episode",
                value=avg_forecasts,
                lower_confidence_bound=avg_forecasts,
                upper_confidence_bound=avg_forecasts,
            ),
        ]


# Run multiple episodes and compute metrics
np.random.seed(42)
metrics_env = WeatherPOMDPWithMetrics(discount_factor=0.95)

pomcp_metrics = POMCP(
    environment=metrics_env,
    discount_factor=0.95,
    depth=10,
    exploration_constant=10.0,
    name="POMCP_Metrics",
    n_simulations=100,
)

histories = []
for ep in range(10):
    b = get_initial_belief(metrics_env, n_particles=30, resampling=True)
    h = run_episode(
        environment=metrics_env,
        policy=pomcp_metrics,
        initial_belief=b,
        num_steps=10,
        logger=None,
    )
    histories.append(h)

metrics = metrics_env.compute_metrics(histories)
print("Custom Metrics (10 episodes):")
for m in metrics:
    print(f"  {m.name}: {m.value:.3f}")

2026-02-25 13:02:57,831 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:296 - Initializing WeatherPOMDP environment with discount factor 0.95
2026-02-25 13:02:57,833 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: POMCP_Metrics (debug=False)


Custom Metrics (10 episodes):
  outdoor_success_rate: 0.767
  avg_forecasts_per_episode: 3.200


## 6. Brief: Continuous Environments

For continuous action/observation spaces, the main differences are:

1. **Inherit from `Environment`** (not `DiscreteActionsEnvironment`) — no `get_actions()` method
2. **Set `SpaceType.CONTINUOUS`** in `SpaceInfo` for the relevant spaces
3. **Use PFT_DPW or POMCPOW** as planners (POMCP requires discrete actions)
4. **Provide an `ActionSampler`** that generates actions from the continuous space

See the `ContinuousLightDarkPOMDP` in the package for a full example.

## Summary: Implementation Checklist

When building a new POMDP environment:

1. **Define your POMDP** — states, actions, observations, dynamics, rewards
2. **Create a `StateTransitionModel`** — implement `sample()` and `probability()`
3. **Create an `ObservationModel`** — implement `sample()` and **`probability()`** (required for weighted particle belief updates)
4. **Assemble the environment class** — inherit from `DiscreteActionsEnvironment` or `Environment`
5. **Test with `sample_next_step()`** — verify dynamics before running planners
6. **Run a planner** — use `run_episode()` to verify end-to-end
7. **(Optional) Add custom metrics** — override `get_metric_names()` and `compute_metrics()`

## What's Next?

- **Basic usage**: See [basic_usage.ipynb](basic_usage.ipynb) for running existing environments with planners
- **Belief representations**: See [belief_representations.ipynb](belief_representations.ipynb) for particle, Gaussian, and GMM beliefs
- **Inspect planner behavior**: See [tree_analysis_debugging.ipynb](tree_analysis_debugging.ipynb) for MCTS tree diagnostics
- **Compare planners at scale**: See [planners_comparison.ipynb](planners_comparison.ipynb) for batch evaluations
- **Tune hyperparameters**: See [hyperparameter_tuning.ipynb](hyperparameter_tuning.ipynb) for optimizing planner parameters with Optuna